## Phần 1: OpenNMT-py - huấn luyện mô hình NMT nhỏ

In [ ]:
!nvidia-smi | head -n 15
!pip install -q OpenNMT-py
!onmt_train --help | head -n 3

Tue Nov 17 03:02:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 1.1 Tải dữ liệu mẫu (English - German, ~10.000 câu)

In [ ]:
!wget -q https://s3.amazonaws.com/opennmt-trainingdata/toy-ende.tar.gz
!tar xf toy-ende.tar.gz
!ls toy-ende
!head -n 3 toy-ende/src-train.txt
!head -n 3 toy-ende/tgt-train.txt

src-test.txt   src-val.txt   tgt-train.txt
src-train.txt  tgt-test.txt  tgt-val.txt
It is not acceptable that , with the help of the national bureaucracies , Parliament &apos;s legislative prerogative should be made null and void by means of implementing provisions whose content , purpose and extent are not laid down in advance .
Federal Master Trainer and Senior Instructor of the Italian Federation of Aerobic Fitness , Group Fitness , Postural Gym , Stretching and Pilates; from 2004 , he has been collaborating with Antiche Terme as personal Trainer and Instructor of Stretching , Pilates and Postural Gym .
&quot; Two soldiers came up to me and told me that if I refuse to sleep with them , they will kill me . They beat me and ripped my clothes .
Es geht nicht an , dass über Ausführungsbestimmungen , deren Inhalt , Zweck und Ausmaß vorher nicht bestimmt ist , zusammen mit den nationalen Bürokratien das Gesetzgebungsrecht des Europäischen Parlaments ausgehebelt wird .
Meistertrainer und l

### 1.2 Tạo file cấu hình

In [ ]:
config = '''
# Dữ liệu
save_data: toy-ende/run/example
src_vocab: toy-ende/run/example.vocab.src
tgt_vocab: toy-ende/run/example.vocab.tgt
overwrite: True

data:
    corpus_1:
        path_src: toy-ende/src-train.txt
        path_tgt: toy-ende/tgt-train.txt
    valid:
        path_src: toy-ende/src-val.txt
        path_tgt: toy-ende/tgt-val.txt

# Huấn luyện
save_model: toy-ende/run/model
save_checkpoint_steps: 1000
train_steps: 3000
valid_steps: 1000
report_every: 100

# GPU
world_size: 1
gpu_ranks: [0]

# Mô hình Transformer cỡ nhỏ
encoder_type: transformer
decoder_type: transformer
enc_layers: 2
dec_layers: 2
heads: 4
hidden_size: 256
word_vec_size: 256
transformer_ff: 1024
dropout: [0.1]
position_encoding: true
batch_size: 1024
batch_type: tokens
optim: adam
learning_rate: 2
warmup_steps: 800
decay_method: noam
param_init_glorot: true
'''
with open('toy_en_de.yaml', 'w') as f:
    f.write(config)

### 1.3 Xây từ vựng và huấn luyện (~vài phút trên GPU)

In [ ]:
!onmt_build_vocab -config toy_en_de.yaml -n_sample 10000
!onmt_train -config toy_en_de.yaml

/bin/bash: line 1: onmt_build_vocab: command not found
/bin/bash: line 1: onmt_train: command not found


### 1.4 Dịch thử

In [ ]:
!onmt_translate -model toy-ende/run/model_step_3000.pt \
    -src toy-ende/src-test.txt -output toy-ende/pred.txt -gpu 0 -verbose 2>&1 | head -n 20

/bin/bash: line 1: onmt_translate: command not found


In [ ]:
!head -n 5 toy-ende/src-test.txt
print('----')
!head -n 5 toy-ende/pred.txt

Orlando Bloom and Miranda Kerr still love each other
Actors Orlando Bloom and Model Miranda Kerr want to go their separate ways .
However , in an interview , Bloom has said that he and Kerr still love each other .
Miranda Kerr and Orlando Bloom are parents to two-year-old Flynn .
Actor Orlando Bloom announced his separation from his wife , supermodel Miranda Kerr .
----
head: cannot open 'toy-ende/pred.txt' for reading: No such file or directory


> **Lưu ý:** dữ liệu chỉ ~10.000 câu và train 3.000 bước nên bản dịch còn kém. Đây là bình thường - mục đích là minh họa quy trình
> *chuẩn bị dữ liệu → xây từ vựng → huấn luyện → dịch*. Phần 2 bên dưới cho chất lượng thực tế.

## Phần 2: CTranslate2 - dịch Anh → Việt bằng mô hình đã huấn luyện sẵn

In [ ]:
!pip install -q ctranslate2 transformers sentencepiece

In [ ]:
# Chuyển mô hình Helsinki-NLP/opus-mt-en-vi sang định dạng CTranslate2
!ct2-transformers-converter --model Helsinki-NLP/opus-mt-en-vi --output_dir opus-mt-en-vi-ct2 --force

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 258/258 [00:00<00:00, 25977.16it/s]
/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [ ]:
import ctranslate2, transformers

translator = ctranslate2.Translator("opus-mt-en-vi-ct2", device="cpu")
tokenizer = transformers.AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-vi")

def translate(sentences):
    tokens = [tokenizer.convert_ids_to_tokens(tokenizer.encode(s)) for s in sentences]
    results = translator.translate_batch(tokens, beam_size=4)
    return [tokenizer.decode(tokenizer.convert_tokens_to_ids(r.hypotheses[0]), skip_special_tokens=True)
            for r in results]

sentences = [
    "Machine translation has evolved from rule-based systems to neural networks.",
    "The Transformer architecture uses self-attention to capture long-range dependencies.",
    "I am studying natural language processing at university.",
]
for s, t in zip(sentences, translate(sentences)):
    print("EN:", s)
    print("VI:", t)
    print()

EN: Machine translation has evolved from rule-based systems to neural networks.
VI: Công nghệ đã tiến hóa từ hệ thống thống thống thống trị sang mạng lưới.

EN: The Transformer architecture uses self-attention to capture long-range dependencies.
VI: Hệ thống biến đổi sử dụng tự hoạt động để bắt giữ các quan hệ phụ thuộc dài hạn.

EN: I am studying natural language processing at university.
VI: Tôi đang học ngôn ngữ tự nhiên ở trường đại học.



### Tự nhập câu để dịch

In [ ]:
text = input("Nhập câu tiếng Anh: ")
print("Tiếng Việt:", translate([text])[0])

Nhập câu tiếng Anh: Hello.How are you?
Tiếng Việt: Xin chào. Cô khỏe không?
